# 03 · Walk-forward dan pelatihan model

**Tujuan:** melatih Regresi Logistik dan XGBoost untuk K1, K2, K3 di tujuh periode uji, tanpa pernah melihat masa depan.

**Input:** `output/02_fitur.parquet`
**Output:** `output/03_prediksi.parquet`, `output/03_parameter.csv`

Padanan di kode penelitian: `folds.py`, `models.py`, `evaluation.py`.

Menjalankan semua periode memakan waktu (XGBoost: 8 kandidat × 3 blok validasi × 7 periode × 3 konfigurasi). Untuk mencoba cepat, isi `PERIODE_DIJALANKAN` dengan satu nama periode.

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

OUTPUT = Path("output")
PERIODE_DIJALANKAN = None      # None = semua, atau misalnya ["2023-H1"]

df = pd.read_parquet(OUTPUT / "02_fitur.parquet")
df = df[df["fitur_layak"] & df["target_layak"]].reset_index(drop=True)   # hanya bar layak
print("Bar layak:", len(df))

Bar layak: 36658


## 1. Konfigurasi fitur, periode uji, dan kisi hiperparameter (dibekukan di Bab 3)

In [2]:
K1 = ["return_lag_0", "return_lag_1", "return_lag_2", "return_lag_3", "return_lag_4",
      "return_lag_5", "log_close_open", "log_high_low", "sigma_24"]
K2 = K1 + ["rsi_14", "macd_line_norm", "macd_signal_norm", "macd_hist_norm"]
K3 = K2 + [c for c in df.columns if c.startswith("macro_")]
KONFIGURASI = {"K1": K1, "K2": K2, "K3": K3}

PERIODE = {   # nama: (awal termasuk, akhir tidak termasuk), UTC
    "2023-H1": ("2023-01-01", "2023-07-01"), "2023-H2": ("2023-07-01", "2024-01-01"),
    "2024-H1": ("2024-01-01", "2024-07-01"), "2024-H2": ("2024-07-01", "2025-01-01"),
    "2025-H1": ("2025-01-01", "2025-07-01"), "2025-H2": ("2025-07-01", "2026-01-01"),
    "2026-Jan-Jul": ("2026-01-01", "2026-08-01"),
}
PERIODE = {k: (pd.Timestamp(a, tz="UTC"), pd.Timestamp(b, tz="UTC")) for k, (a, b) in PERIODE.items()}

KISI = {
    "logistic_regression": [{"C": c} for c in (0.01, 0.1, 1.0, 10.0)],
    "xgboost": [{"max_depth": d, "n_estimators": n, "learning_rate": lr}
                for d in (3, 5) for n in (100, 300) for lr in (0.05, 0.1)],
}
KELAS = ["SELL", "HOLD", "BUY"]     # kode kelas 0, 1, 2

## 2. Pembagian kronologis dan purge

- **Data uji:** keputusan di dalam periode **dan** label selesai sebelum periode berakhir.
- **Data latih:** semua bar yang labelnya selesai **sebelum** keputusan pertama periode uji. Bar latih terakhir yang labelnya jatuh di periode uji ikut terbuang: inilah **purge**.
- **Validasi internal:** 18 bulan sebelum periode uji dibagi tiga blok 6 bulan. Tiap blok dilatih dengan semua data sebelumnya.

In [3]:
def mask_latih(batas):
    return df["target_end_time"] < batas

def mask_uji(awal, akhir):
    return (df["decision_time"] >= awal) & (df["decision_time"] < akhir) & (df["target_end_time"] < akhir)

def blok_validasi(awal_uji):
    mulai = awal_uji - pd.DateOffset(months=18)
    return [(mulai + pd.DateOffset(months=6 * i), mulai + pd.DateOffset(months=6 * (i + 1))) for i in range(3)]

ringkas = []
for nama, (awal, akhir) in PERIODE.items():
    uji = mask_uji(awal, akhir)
    ringkas.append({"periode": nama, "bar_latih": mask_latih(awal).sum(), "bar_uji": uji.sum(),
                    **df.loc[uji, "label"].value_counts().reindex(["BUY", "HOLD", "SELL"]).to_dict()})
pd.DataFrame(ringkas)

,periode,bar_latih,bar_uji,BUY,HOLD,SELL
0,2023-H1,16756,2807,699,1410,698
1,2023-H2,19563,2829,669,1470,690
2,2024-H1,22393,2808,740,1385,683
3,2024-H2,25201,2870,830,1380,660
4,2025-H1,28071,2735,816,1229,690
5,2025-H2,30807,2746,838,1219,689
6,2026-Jan-Jul,33554,3103,809,1501,793


## 3. Praproses yang hanya belajar dari data latih

1. Kolom yang seluruhnya kosong di data latih dibuang.
2. Nilai kosong diisi **median data latih**, ditambah kolom indikator "nilai ini kosong".
3. Standardisasi (mean/std data latih) untuk Logistik, dan untuk XGBoost hanya pada K3.

Data uji tidak pernah ikut menghitung median atau mean.

In [4]:
def praproses(latih, uji, kolom, skala):
    dipakai = [c for c in kolom if latih[c].notna().any()]
    median = latih[dipakai].median()
    ada_kosong = [c for c in dipakai if latih[c].isna().any()]

    def matriks(d):
        return np.column_stack([d[dipakai].fillna(median).to_numpy(float), d[ada_kosong].isna().to_numpy(float)])

    x_latih, x_uji = matriks(latih), matriks(uji)
    if skala:
        scaler = StandardScaler().fit(x_latih)
        x_latih, x_uji = scaler.transform(x_latih), scaler.transform(x_uji)
    return x_latih, x_uji

## 4. Latih satu model, prediksi data uji

- **Logistik:** lbfgs, L2, maksimum 2.000 iterasi, `class_weight="balanced"`. Kandidat yang tidak konvergen dianggap gagal.
- **XGBoost:** `multi:softprob`, subsample 0,8, colsample 0,8, λ = 1, seed 42, bobot kelas n / (3·n_kelas).

In [5]:
def latih_prediksi(model, param, latih, uji, kolom):
    y = latih["label"].map({k: i for i, k in enumerate(KELAS)}).to_numpy()
    if set(y) != {0, 1, 2}:
        raise ValueError("data latih tidak memuat tiga kelas")
    skala = model == "logistic_regression" or kolom is K3
    x_latih, x_uji = praproses(latih, uji, kolom, skala)

    if model == "logistic_regression":
        m = LogisticRegression(C=param["C"], solver="lbfgs", max_iter=2000, class_weight="balanced")
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            m.fit(x_latih, y)
    else:
        bobot = len(y) / (3 * np.bincount(y, minlength=3)[y])
        m = XGBClassifier(objective="multi:softprob", num_class=3, subsample=0.8, colsample_bytree=0.8,
                          reg_lambda=1.0, random_state=42, n_jobs=1, tree_method="hist",
                          eval_metric="mlogloss", **param)
        m.fit(x_latih, y, sample_weight=bobot)
    return np.array(KELAS)[m.predict(x_uji).astype(int)]


def macro_f1(benar, prediksi):
    return f1_score(benar, prediksi, labels=KELAS, average="macro", zero_division=0)

## 5. Pilih hiperparameter dengan validasi internal

Setiap kandidat dinilai dengan rata-rata Macro-F1 tiga blok validasi. Kandidat yang gagal di salah satu blok tidak ikut. Jika skor sama: Logistik memilih C lebih kecil; XGBoost memilih kedalaman, lalu jumlah iterasi, lalu learning rate yang lebih kecil.

In [6]:
def pilih_parameter(model, kolom, awal_uji):
    hasil = []
    for param in KISI[model]:
        skor = []
        try:
            for awal, akhir in blok_validasi(awal_uji):
                latih, valid = df[mask_latih(awal)], df[mask_uji(awal, akhir)]
                skor.append(macro_f1(valid["label"], latih_prediksi(model, param, latih, valid, kolom)))
        except (ValueError, RuntimeError, ConvergenceWarning):
            continue
        hasil.append((np.mean(skor), param))
    urut = lambda h: (-h[0], *h[1].values()) if model == "logistic_regression" else \
                     (-h[0], h[1]["max_depth"], h[1]["n_estimators"], h[1]["learning_rate"])
    return min(hasil, key=urut)[1]

## 6. Jalankan eksperimen

Bar uji sama untuk K1, K2, K3 dan kedua model. Skor periode uji tidak dipakai untuk memilih apa pun.

In [7]:
prediksi, parameter = [], []
mulai = time.time()
for nama, (awal, akhir) in PERIODE.items():
    if PERIODE_DIJALANKAN and nama not in PERIODE_DIJALANKAN:
        continue
    latih, uji = df[mask_latih(awal)], df[mask_uji(awal, akhir)]
    for model in ("logistic_regression", "xgboost"):
        for konfig, kolom in KONFIGURASI.items():
            param = pilih_parameter(model, kolom, awal)
            hasil = latih_prediksi(model, param, latih, uji, kolom)
            prediksi.append(pd.DataFrame({"periode": nama, "model": model, "konfigurasi": konfig,
                                          "time": uji["time"].to_numpy(), "benar": uji["label"].to_numpy(),
                                          "prediksi": hasil}))
            parameter.append({"periode": nama, "model": model, "konfigurasi": konfig, **param,
                              "macro_f1": macro_f1(uji["label"], hasil)})
            print(f"{nama:13} {model:20} {konfig}  {param}  macro-F1 = {parameter[-1]['macro_f1']:.6f}"
                  f"  ({time.time() - mulai:.0f} dtk)")

2023-H1       logistic_regression  K1  {'C': 0.01}  macro-F1 = 0.423792  (0 dtk)


2023-H1       logistic_regression  K2  {'C': 0.01}  macro-F1 = 0.423555  (1 dtk)


2023-H1       logistic_regression  K3  {'C': 10.0}  macro-F1 = 0.413261  (1 dtk)


2023-H1       xgboost              K1  {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  macro-F1 = 0.424583  (8 dtk)


2023-H1       xgboost              K2  {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  macro-F1 = 0.413318  (19 dtk)


2023-H1       xgboost              K3  {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  macro-F1 = 0.427637  (37 dtk)


... (32 outputs left out)

2026-Jan-Jul  logistic_regression  K3  {'C': 0.01}  macro-F1 = 0.379819  (292 dtk)


2026-Jan-Jul  xgboost              K1  {'max_depth': 3, 'n_estimators': 100, 'learning_rate': 0.05}  macro-F1 = 0.367244  (306 dtk)


2026-Jan-Jul  xgboost              K2  {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.05}  macro-F1 = 0.385605  (322 dtk)


2026-Jan-Jul  xgboost              K3  {'max_depth': 3, 'n_estimators': 300, 'learning_rate': 0.05}  macro-F1 = 0.383374  (356 dtk)


In [8]:
pd.concat(prediksi).to_parquet(OUTPUT / "03_prediksi.parquet", index=False)
pd.DataFrame(parameter).to_csv(OUTPUT / "03_parameter.csv", index=False)
print("Tersimpan:", OUTPUT / "03_prediksi.parquet", "dan", OUTPUT / "03_parameter.csv")

Tersimpan: output/03_prediksi.parquet dan output/03_parameter.csv
